In [1]:
import torch
import torch.nn as nn
import timm
import pandas as pd
import numpy as np
import os
import math
import json
from PIL import Image
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split, StratifiedKFold
from tqdm import tqdm
import random

# ===================== SEED FÜR REPRODUZIERBARKEIT =====================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ===================== DATEN LADEN =====================
data_dir = "/datasets/multi-view-pig-posture-recognition/"
train2_df = pd.read_csv(os.path.join(data_dir, "train2.csv"))
dir_train2 = os.path.join(data_dir, "train2_images")
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))
dir_test = os.path.join(data_dir, "test_images")

def parse_bbox_safe(x):
    if isinstance(x, str):
        return [float(i) for i in x.replace("[", "").replace("]", "").split(",")]
    return x

train2_df["bbox"] = train2_df["bbox"].apply(parse_bbox_safe)
test_df["bbox"] = test_df["bbox"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
train2_df["camera_id"] = train2_df["image_id"].apply(lambda x: x.split("_")[0])

print(f"Train2: {len(train2_df)} samples, Test: {len(test_df)} samples")
print(train2_df["class_id"].value_counts().sort_index())

<jemalloc>: Unsupported system page size


Device: cuda:3
Train2: 23450 samples, Test: 11708 samples
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64


In [2]:
# ===================== DATASET MIT MIXUP =====================
class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, use_mixup=False, alpha=0.2):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.use_mixup = use_mixup
        self.alpha = alpha
        self.num_classes = 5

    def __len__(self):
        return len(self.df)

    def _load_image(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")
        bbox = row["bbox"]
        x, y, w, h = bbox
        img_w, img_h = image.size
        x_min = max(0, int(x))
        y_min = max(0, int(y))
        x_max = min(img_w, int(x + w))
        y_max = min(img_h, int(y + h))
        if x_max > x_min and y_max > y_min:
            image = image.crop((x_min, y_min, x_max, y_max))
        label = int(row["class_id"])
        return image, label

    def __getitem__(self, idx):
        image, label = self._load_image(idx)
        if self.transform:
            image = self.transform(image)

        if self.use_mixup and random.random() < 0.3:
            idx2 = random.randint(0, len(self.df) - 1)
            image2, label2 = self._load_image(idx2)
            if self.transform:
                image2 = self.transform(image2)
            lam = np.random.beta(self.alpha, self.alpha)
            image = lam * image + (1 - lam) * image2
            # Soft labels für Mixup
            soft_label = torch.zeros(self.num_classes)
            soft_label[label] = lam
            soft_label[label2] += (1 - lam)
            return image, soft_label

        return image, label

print("✅ Dataset mit Mixup-Support ready")

✅ Dataset mit Mixup-Support ready


In [3]:
# ===================== TRANSFORMS - STÄRKER & GRÖSSER =====================
IMG_SIZE = 384  # Größer als vorher (224) → viel besser für ConvNeXt!

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15), shear=10),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# TTA-Transforms für Test Time Augmentation
tta_transforms = [
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation((90, 90)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
]

print(f"✅ Transforms ready | IMG_SIZE={IMG_SIZE} | TTA={len(tta_transforms)} augmentations")

✅ Transforms ready | IMG_SIZE=384 | TTA=5 augmentations


In [4]:
# ===================== TRAIN/VAL SPLIT =====================
train_df, val_df = train_test_split(
    train2_df, test_size=0.15, stratify=train2_df["class_id"], random_state=42
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

# Klassen-Gewichte mit stärkerem Fokus auf Minderheitsklassen
train_labels = train_df["class_id"].tolist()
class_counts = Counter(train_labels)
num_classes = len(class_counts)
total = sum(class_counts.values())

# Inverse frequency (stärker als sqrt)
class_weights = []
for i in range(num_classes):
    w = (total / (num_classes * class_counts[i])) ** 0.75  # 0.75 Exponent
    class_weights.append(w)
    print(f"Klasse {i}: {class_counts[i]} samples, weight={w:.3f}")

class_weights_tensor = torch.FloatTensor(class_weights).to(device)
# Label smoothing 0.1 statt 0.05
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

BATCH_SIZE = 16  # Kleiner wegen größerer Bilder (384x384)

train_dataset = PigPostureDataset(train_df, dir_train2, transform=train_transform, use_mixup=True)
val_dataset = PigPostureDataset(val_df, dir_train2, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

print(f"\n✅ Train: {len(train_loader)} Batches, Val: {len(val_loader)} Batches")

Train: 19932, Val: 3518
Klasse 0: 2620 samples, weight=1.370
Klasse 1: 2920 samples, weight=1.263
Klasse 2: 591 samples, weight=4.185
Klasse 3: 8439 samples, weight=0.570
Klasse 4: 5362 samples, weight=0.801

✅ Train: 1245 Batches, Val: 220 Batches


In [8]:
class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, use_mixup=False, alpha=0.2):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.use_mixup = use_mixup
        self.alpha = alpha
        self.num_classes = 5

    def __len__(self):
        return len(self.df)

    def _load_image(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")
        bbox = row["bbox"]
        x, y, w, h = bbox
        img_w, img_h = image.size
        x_min = max(0, int(x))
        y_min = max(0, int(y))
        x_max = min(img_w, int(x + w))
        y_max = min(img_h, int(y + h))
        if x_max > x_min and y_max > y_min:
            image = image.crop((x_min, y_min, x_max, y_max))
        label = int(row["class_id"])
        return image, label

    def __getitem__(self, idx):
        image, label = self._load_image(idx)
        if self.transform:
            image = self.transform(image)

        if self.use_mixup and random.random() < 0.3:
            idx2 = random.randint(0, len(self.df) - 1)
            image2, label2 = self._load_image(idx2)
            if self.transform:
                image2 = self.transform(image2)
            lam = np.random.beta(self.alpha, self.alpha)
            image = lam * image + (1 - lam) * image2
            soft_label = torch.zeros(self.num_classes)
            soft_label[label] = lam
            soft_label[label2] += (1 - lam)
            return image, soft_label  # Tensor ✅

        # ✅ FIX: Immer Tensor zurückgeben, nicht int!
        hard_label = torch.zeros(self.num_classes)
        hard_label[label] = 1.0
        return image, hard_label

print("✅ Dataset fixed")

✅ Dataset fixed


In [9]:
# ===================== MIXUP-KOMPATIBLER LOSS =====================
def mixup_criterion(criterion, outputs, targets):
    """Targets sind jetzt immer Soft-Label Tensoren (2D)"""
    # Soft labels: manueller Cross-Entropy (funktioniert auch für One-Hot)
    log_probs = torch.log_softmax(outputs, dim=1)
    loss = -(targets * log_probs).sum(dim=1).mean()
    return loss

print("✅ Loss function fixed")

# ===================== VERBESSERTE TRAININGSSCHLEIFE =====================
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, max_epochs, min_lr_ratio=0.01):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr_ratio = min_lr_ratio
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        if self.current_epoch <= self.warmup_epochs:
            lr_scale = self.current_epoch / self.warmup_epochs
        else:
            progress = (self.current_epoch - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)
            lr_scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = pg['initial_lr'] * lr_scale

    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

def setup_scheduler(optimizer, warmup, max_ep):
    for pg in optimizer.param_groups:
        pg['initial_lr'] = pg['lr']
    return CosineWarmupScheduler(optimizer, warmup, max_ep)


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=10, save_name="best_model.pth", accumulation_steps=4):
    best_acc = 0.0
    patience = 0
    max_patience = 5  # Early stopping

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        for i, (images, labels) in enumerate(tqdm(train_loader, desc=f"Ep {epoch+1}/{num_epochs} Train")):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, labels) / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item() * accumulation_steps

        # Validation
        model.eval()
        correct, total_val = 0, 0
        val_loss = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Ep {epoch+1}/{num_epochs} Val"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        train_loss = running_loss / len(train_loader)
        val_loss = val_loss / len(val_loader)
        val_acc = correct / total_val
        lr = scheduler.get_last_lr()[0]
        scheduler.step()

        # Per-Klasse Accuracy
        from sklearn.metrics import classification_report, f1_score
        f1 = f1_score(all_labels, all_preds, average='macro')

        saved = ""
        if val_acc > best_acc:
            best_acc = val_acc
            patience = 0
            torch.save(model.state_dict(), save_name)
            saved = " ✅ SAVED"
        else:
            patience += 1

        print(f"Ep {epoch+1} | TrL: {train_loss:.4f} | VaL: {val_loss:.4f} | "
              f"Acc: {val_acc:.4f} | F1: {f1:.4f} | LR: {lr:.2e}{saved}")

        if patience >= max_patience:
            print(f"⚠️ Early stopping nach {epoch+1} Epochs")
            break

    return best_acc

print("✅ Training functions ready")

✅ Loss function fixed
✅ Training functions ready


In [1]:
# ===================== PHASE 1: HEAD TRAINING =====================
optimizer = torch.optim.AdamW(model.head.parameters(), lr=3e-3, weight_decay=1e-2)
scheduler = setup_scheduler(optimizer, warmup=2, max_ep=8)

print("=== PHASE 1: Train Head Only ===")
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                       num_epochs=1, save_name="best_phase1.pth", accumulation_steps=4)
print(f"\n🏆 Phase 1 Best Acc: {best_acc:.4f}")

NameError: name 'torch' is not defined

In [ ]:
# ===================== PHASE 2: FINE-TUNE 50% BACKBONE =====================
model.load_state_dict(torch.load("best_phase1.pth", map_location=device))
model.unfreeze_backbone(fraction=0.5)

optimizer_ft = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 5e-6},
    {"params": model.head.parameters(), "lr": 2e-5},
], weight_decay=1e-2)
scheduler_ft = setup_scheduler(optimizer_ft, warmup=2, max_ep=10)

print("=== PHASE 2: Fine-Tune Backbone (50%) ===")
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer_ft, scheduler_ft,
                       num_epochs=10, save_name="best_phase2.pth", accumulation_steps=4)
print(f"\n🏆 Phase 2 Best Acc: {best_acc:.4f}")

In [ ]:
# ===================== PHASE 3: FULL FINE-TUNE (NEU!) =====================
model.load_state_dict(torch.load("best_phase2.pth", map_location=device))
model.unfreeze_all()

optimizer_full = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters()], "lr": 1e-6},
    {"params": model.head.parameters(), "lr": 5e-6},
], weight_decay=5e-3)
scheduler_full = setup_scheduler(optimizer_full, warmup=1, max_ep=8)

print("=== PHASE 3: Full Fine-Tune ===")
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer_full, scheduler_full,
                       num_epochs=8, save_name="best_phase3.pth", accumulation_steps=4)
print(f"\n🏆 Phase 3 Best Acc: {best_acc:.4f}")

In [ ]:
# ===================== TEST TIME AUGMENTATION (TTA) INFERENCE =====================
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")
        bbox = row["bbox"]
        if isinstance(bbox, str):
            bbox = json.loads(bbox)
        x, y, w, h = bbox
        img_w, img_h = image.size
        x_min = max(0, int(x))
        y_min = max(0, int(y))
        x_max = min(img_w, int(x + w))
        y_max = min(img_h, int(y + h))
        if x_max > x_min and y_max > y_min:
            image = image.crop((x_min, y_min, x_max, y_max))
        image = self.transform(image)
        return image, row["row_id"]


def predict_with_tta(model, test_df, dir_test, tta_transforms, device, batch_size=32):
    """Führt TTA durch und mittelt die Wahrscheinlichkeiten"""
    model.eval()
    all_probs = None
    all_row_ids = None

    for t_idx, transform in enumerate(tta_transforms):
        print(f"TTA {t_idx+1}/{len(tta_transforms)}...")
        test_dataset = TestDataset(test_df, dir_test, transform)
        test_loader = DataLoader(test_dataset, batch_size=batch_size,
                                 shuffle=False, num_workers=4)

        probs_list = []
        row_ids_list = []

        with torch.no_grad():
            for images, rids in tqdm(test_loader, desc=f"  TTA {t_idx+1}"):
                images = images.to(device)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)
                probs_list.append(probs.cpu().numpy())
                if t_idx == 0:
                    row_ids_list.extend(list(rids))

        probs_arr = np.vstack(probs_list)

        if all_probs is None:
            all_probs = probs_arr
            all_row_ids = row_ids_list
        else:
            all_probs += probs_arr

    all_probs /= len(tta_transforms)
    predictions = np.argmax(all_probs, axis=1)
    return all_row_ids, predictions, all_probs


# ===================== AUTOMATISCHE MODELLAUSWAHL =====================
# Wählt automatisch das beste verfügbare Modell (Phase 3 > Phase 2 > Phase 1)
model_candidates = [
    ("best_phase3.pth", "Phase 3 - Full Fine-Tune"),
    ("best_phase2.pth", "Phase 2 - Backbone 50%"),
    ("best_phase1.pth", "Phase 1 - Head Only"),
]

best_model_path = None
for path, desc in model_candidates:
    if os.path.exists(path):
        best_model_path = path
        print(f"✅ Verwende Modell: {path} ({desc})")
        break

if best_model_path is None:
    raise FileNotFoundError("❌ Kein gespeichertes Modell gefunden! Training zuerst ausführen.")

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()
print(f"📦 Modell geladen: {best_model_path}")

print("\n=== TTA INFERENCE ===")
row_ids, predictions, probs = predict_with_tta(
    model, test_df, dir_test, tta_transforms, device, batch_size=32
)

submission = pd.DataFrame({"row_id": row_ids, "class_id": predictions})
submission.to_csv("improved_submission.csv", index=False)

print(f"\n✅ Gespeichert als: improved_submission.csv")
print(f"📊 Modell: {best_model_path}")
print(f"\nKlassenverteilung:")
print(submission["class_id"].value_counts().sort_index())
print(submission.head())